# Strategy lifecycle causal screens — 2026-07-15

## TL;DR

All three fixed causal mechanisms were rejected. Two-second entry persistence removed 13 of 17 known winners. A fee-inclusive executable profit lock improved toxic-fold PnL by $2.91 but made three terminal winners negative. A 30-second pre-close loss cut improved toxic-fold PnL by $7.08 but made two terminal winners negative. No strategy parameters were changed, and `live_ready` remains false.

## Context and decision

The strict-42 volatility-floor baseline is profitable in aggregate but fails the fixed A+ Wilson, profitable-report, and clustered-loss gates. This notebook tests whether three predeclared causal lifecycle mechanisms can remove the folds 29–41 loss cluster without creating new losses. Aggregate PnL improvement alone is not a pass.

## Data and definitions

The durable evidence file combines the exact strict-42 baseline trades, one-Hz post-gate opportunity rows, and target-token PMXT v2 book/change streams. Exit marks include both entry and hypothetical exit fees, use full-size visible-bid FOK quotes, and apply the measured 202 ms order insertion latency. The lifecycle results are causal reconstructions, not integrated harness variants; exact engine replay was intentionally skipped because each causal screen failed first.

In [1]:
import json
from pathlib import Path

evidence_path = Path('deploy/promotions/evidence/strategy_registry/20260715_strategy_lifecycle_causal_screens.json')
evidence = json.loads(evidence_path.read_text())
baseline = evidence['baseline']
print({
    'trades': baseline['trades'],
    'total_pnl_usd': baseline['total_pnl_usd'],
    'wilson_lower_bound': baseline['wilson_lower_bound'],
    'profitable_reports': f"{baseline['profitable_eligible_reports']}/{baseline['eligible_reports']}",
    'loss_burst': baseline['loss_burst'],
    'a_plus_pass': baseline['a_plus_pass'],
})

{'trades': 102, 'total_pnl_usd': 13.54, 'wilson_lower_bound': 0.6843, 'profitable_reports': '19/39', 'loss_burst': 4, 'a_plus_pass': False}


## Results

The screen table below is deliberately pass/fail oriented. A mechanism that improves aggregate tail PnL still fails if it removes or turns a known winner negative.

In [2]:
screen_rows = []
for screen in evidence['screens']:
    result = screen['results']
    screen_rows.append({
        'screen': screen['name'],
        'status': screen['status'],
        'harmed_or_removed_winners': result.get('harmed_winners', result.get('removed_wins')),
        'tail_delta_pnl_usd': result.get('delta_pnl_usd'),
        'counterfactual_profitable_folds': result.get('counterfactual_profitable_folds'),
    })
screen_rows

[{'screen': 'two_second_entry_persistence',
  'status': 'reject',
  'harmed_or_removed_winners': 13,
  'tail_delta_pnl_usd': None,
  'counterfactual_profitable_folds': None},
 {'screen': 'executable_profit_lock',
  'status': 'reject',
  'harmed_or_removed_winners': 3,
  'tail_delta_pnl_usd': 2.91237,
  'counterfactual_profitable_folds': 5},
 {'screen': 'thirty_second_late_loss_cut',
  'status': 'reject',
  'harmed_or_removed_winners': 2,
  'tail_delta_pnl_usd': 7.08078,
  'counterfactual_profitable_folds': 5}]

In [3]:
fold_comparison = [
    {
        'fold': row['fold'],
        'trades': row['trades'],
        'baseline': row['baseline_pnl_usd'],
        'profit_lock': row['profit_lock_pnl_usd'],
        'late_loss_cut': row['late_loss_cut_pnl_usd'],
    }
    for row in evidence['fold_comparison']
]
fold_comparison

[{'fold': 29,
  'trades': 6,
  'baseline': -0.24369,
  'profit_lock': -1.85157,
  'late_loss_cut': -2.27184},
 {'fold': 30,
  'trades': 3,
  'baseline': 4.50786,
  'profit_lock': 2.73596,
  'late_loss_cut': 4.50786},
 {'fold': 31,
  'trades': 2,
  'baseline': -3.53932,
  'profit_lock': 1.88408,
  'late_loss_cut': -3.53932},
 {'fold': 32,
  'trades': 1,
  'baseline': -5.01516,
  'profit_lock': -5.01516,
  'late_loss_cut': -2.05036},
 {'fold': 33,
  'trades': 2,
  'baseline': -3.60591,
  'profit_lock': -5.0065,
  'late_loss_cut': -3.60591},
 {'fold': 34,
  'trades': 1,
  'baseline': -5.07776,
  'profit_lock': -5.07776,
  'late_loss_cut': -3.415},
 {'fold': 35,
  'trades': 5,
  'baseline': 6.28189,
  'profit_lock': 2.61903,
  'late_loss_cut': 6.28189},
 {'fold': 36,
  'trades': 3,
  'baseline': -2.96842,
  'profit_lock': -2.96842,
  'late_loss_cut': 0.68464},
 {'fold': 37,
  'trades': 2,
  'baseline': -4.13251,
  'profit_lock': -0.10581,
  'late_loss_cut': -2.88458},
 {'fold': 38,
  'trad

## Robustness and limitations

Fixed 5/15/30/60/90-second executable marks showed no clean after-entry rescue boundary: even at 90 seconds, 3 of 29 available terminal winners were negative while 7 of 10 terminal losses were negative. The analysis therefore rejects deadline tuning on this cohort. It also refuses neighboring profit floors or pre-close horizons after their registered rules failed.

In [4]:
assert evidence['live_ready'] is False
assert evidence['status'] == 'KEEP_REPLAY_RESEARCH'
assert all(screen['status'] == 'reject' for screen in evidence['screens'])
assert evidence['verdict']['strategy_adjustment'] == 'none'
print('Fail-closed assertions passed.')

Fail-closed assertions passed.


## Takeaways

1. Do not implement entry persistence, profit lock, or the 30-second late loss cut.
2. Do not tune neighboring lifecycle thresholds on folds 29–41.
3. The next strategy mechanism must add new causal information or materially different payoff geometry and must be validated on the freshest fully resolved window before promotion credit.
4. The current research system remains A-; the strategy is not A+ and is not live-ready.